## 1. Carga y preparación de datos

En esta sección se carga el conjunto de datos preparado durante la ingeniería de características. Se verifica el orden cronológico, la disponibilidad de las variables objetivo y la consistencia de la información que se utilizará para entrenar y comparar los modelos.


In [2]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Cargar los datos preparados
df_model = pd.read_csv(
    "../data/Silver/grupo_exito_features.csv",
    parse_dates=["fecha"]
)

# Orden cronológico
df_model = (
    df_model
    .sort_values("fecha")
    .reset_index(drop=True)
)

# Variables que queremos pronosticar
variables_objetivo = [
    "ingresos",
    "utilidad_bruta",
    "ebitda_recurrente",
    "resultado_neto"
]

print("Dimensiones del dataset:", df_model.shape)
print("\nPrimera fecha:", df_model["fecha"].min())
print("Última fecha:", df_model["fecha"].max())

print("\nValores faltantes:", df_model.isna().sum().sum())

print("\nVariables objetivo:")
print(df_model[variables_objetivo].describe().round(2))

Dimensiones del dataset: (20, 31)

Primera fecha: 2021-03-31 00:00:00
Última fecha: 2025-12-31 00:00:00

Valores faltantes: 0

Variables objetivo:
         ingresos  utilidad_bruta  ebitda_recurrente  resultado_neto
count       20.00           20.00              20.00           20.00
mean   5127650.70      1313760.15          420134.00        67332.25
std     689673.84       177636.07          111257.93        83608.96
min    3696687.00       959633.00          302113.00       -77668.00
25%    4985491.50      1256620.50          342118.50        -9321.75
50%    5218687.00      1317004.50          379216.50        63401.50
75%    5407315.50      1386000.50          470940.00       130462.50
max    6288024.00      1624970.00          670615.00       212665.00


## 2. Definición del enfoque de modelado

Se compararán modelos de complejidad progresiva para determinar si aportan mejoras frente al baseline estacional.

La evaluación respetará el orden temporal de los datos: las observaciones utilizadas para entrenar cada modelo deben ser anteriores a las observaciones evaluadas.

Dado que la serie original contiene 24 trimestres y el conjunto de características dispone de 20 observaciones, se priorizarán modelos sencillos y se limitará el número de parámetros para reducir el riesgo de sobreajuste.


### 2.1 Definición de los modelos

Se utilizarán los siguientes enfoques:

1. **Baseline estacional (Lag 4):** utiliza como pronóstico el valor observado en el mismo trimestre del año anterior. Se conservará como referencia.
2. **Regresión lineal:** utilizará variables temporales, rezagos y características estacionales para estimar el comportamiento de cada variable financiera.
3. **Suavizamiento exponencial:** se evaluará como alternativa para capturar la evolución temporal y los patrones estacionales de las series.


### 2.3 División temporal Train/Test

Se utilizarán las primeras 16 observaciones para entrenar los modelos y las últimas 4 observaciones para la evaluación, correspondientes a los cuatro trimestres de 2025.

Esta división conserva el orden cronológico y evita utilizar observaciones futuras para entrenar los modelos que se evalúan sobre 2025.


In [3]:
# Definir el período de prueba
n_test = 4

df_train = df_model.iloc[:-n_test].copy()
df_test = df_model.iloc[-n_test:].copy()

print("Observaciones de entrenamiento:", len(df_train))
print("Observaciones de prueba:", len(df_test))

print("\nPeríodo de entrenamiento:")
print(df_train["fecha"].min(), "a", df_train["fecha"].max())

print("\nPeríodo de prueba:")
print(df_test["fecha"].min(), "a", df_test["fecha"].max())

Observaciones de entrenamiento: 16
Observaciones de prueba: 4

Período de entrenamiento:
2021-03-31 00:00:00 a 2024-12-31 00:00:00

Período de prueba:
2025-03-31 00:00:00 a 2025-12-31 00:00:00


### Implementación del modelo de regresión lineal

In [4]:

from sklearn.linear_model import LinearRegression

# Variables estacionales creadas en Feature Engineering
columnas_estacionales = [
    col for col in df_model.columns
    if col in ["T_2", "T_3", "T_4"]
]

print("Variables estacionales:", columnas_estacionales)

Variables estacionales: ['T_2', 'T_3', 'T_4']


In [5]:
# Entrenamiento y generación de pronosticos

resultados_regresion = {}

for variable in variables_objetivo:

    # Características disponibles antes de observar el resultado
    columnas_features = [
        "t",
        f"{variable}_lag1",
        f"{variable}_lag4"
    ] + columnas_estacionales

    X_train = df_train[columnas_features]
    y_train = df_train[variable]

    X_test = df_test[columnas_features]

    # Entrenar el modelo
    modelo = LinearRegression()
    modelo.fit(X_train, y_train)

    # Pronósticos para el período de prueba
    y_pred = modelo.predict(X_test)

    resultados_regresion[variable] = {
        "modelo": modelo,
        "columnas": columnas_features,
        "pronosticos": y_pred
    }

    print(f"\nVariable: {variable}")
    print("Características:", columnas_features)
    print("Modelo entrenado correctamente.")


Variable: ingresos
Características: ['t', 'ingresos_lag1', 'ingresos_lag4', 'T_2', 'T_3', 'T_4']
Modelo entrenado correctamente.

Variable: utilidad_bruta
Características: ['t', 'utilidad_bruta_lag1', 'utilidad_bruta_lag4', 'T_2', 'T_3', 'T_4']
Modelo entrenado correctamente.

Variable: ebitda_recurrente
Características: ['t', 'ebitda_recurrente_lag1', 'ebitda_recurrente_lag4', 'T_2', 'T_3', 'T_4']
Modelo entrenado correctamente.

Variable: resultado_neto
Características: ['t', 'resultado_neto_lag1', 'resultado_neto_lag4', 'T_2', 'T_3', 'T_4']
Modelo entrenado correctamente.


In [6]:
# Comparación de los valores reales y los estimados

comparacion_regresion = df_test[["fecha"]].copy()

for variable in variables_objetivo:
    comparacion_regresion[f"{variable}_real"] = (
        df_test[variable].to_numpy()
    )

    comparacion_regresion[f"{variable}_regresion"] = (
        resultados_regresion[variable]["pronosticos"]
    )

display(comparacion_regresion.round(2))

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_19828\3638352065.py:14: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(comparacion_regresion.round(2))


,fecha,ingresos_real,ingresos_regresion,utilidad_bruta_real,utilidad_bruta_regresion,ebitda_recurrente_real,ebitda_recurrente_regresion,resultado_neto_real,resultado_neto_regresion
16,2025-03-31,5404642,5939930.12,1382773,1588245.73,371148,297271.38,93147,-27243.32
17,2025-06-30,5208469,5652230.27,1335783,1422355.78,452242,335713.23,146865,-59353.44
18,2025-09-30,5228905,5693529.21,1312056,1468346.82,448017,316371.57,142905,-44884.29
19,2025-12-31,6166344,6116240.36,1612428,1677138.67,670615,561843.66,209191,-16050.91


In [7]:
# Creación de metricas del modelo


from sklearn.metrics import mean_absolute_error, mean_squared_error

resultados_metricas_regresion = []

for variable in variables_objetivo:
    y_real = df_test[variable].to_numpy()
    y_pred = resultados_regresion[variable]["pronosticos"]

    mae = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))

    if variable != "resultado_neto":
        mape = np.mean(
            np.abs((y_real - y_pred) / y_real)
        ) * 100
    else:
        mape = np.nan

    resultados_metricas_regresion.append({
        "variable": variable,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE_pct": mape
    })

metricas_regresion = pd.DataFrame(
    resultados_metricas_regresion
)

display(metricas_regresion.round(2))

,variable,MAE,RMSE,MAPE_pct
0,ingresos,373444.31,418880.49,7.03
1,utilidad_bruta,128261.75,139935.86,9.32
2,ebitda_recurrente,107705.54,109770.71,22.82
3,resultado_neto,184909.99,189089.01,NaN


In [8]:
# Recuperación de métricas del modelo baseline

from sklearn.metrics import mean_absolute_error, mean_squared_error

resultados_baseline = []

for variable in variables_objetivo:
    y_real = df_test[variable].to_numpy()
    y_pred = df_test[f"{variable}_lag4"].to_numpy()

    mae = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))

    if variable != "resultado_neto":
        mape = np.mean(
            np.abs((y_real - y_pred) / y_real)
        ) * 100
    else:
        mape = np.nan

    resultados_baseline.append({
        "variable": variable,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE_pct": mape
    })

metricas_baseline = pd.DataFrame(resultados_baseline)

display(metricas_baseline.round(2))

,variable,MAE,RMSE,MAPE_pct
0,ingresos,99564.75,111350.89,1.80
1,utilidad_bruta,33779.00,38135.53,2.46
2,ebitda_recurrente,79396.75,85419.10,17.86
3,resultado_neto,134330.50,141527.88,NaN


In [9]:
# Comparación modelo baseline vs regresión lineal


comparacion_modelos = metricas_baseline.merge(
    metricas_regresion,
    on="variable",
    suffixes=("_baseline", "_regresion")
)

display(comparacion_modelos.round(2))

,variable,MAE_baseline,RMSE_baseline,MAPE_pct_baseline,MAE_regresion,RMSE_regresion,MAPE_pct_regresion
0,ingresos,99564.75,111350.89,1.80,373444.31,418880.49,7.03
1,utilidad_bruta,33779.00,38135.53,2.46,128261.75,139935.86,9.32
2,ebitda_recurrente,79396.75,85419.10,17.86,107705.54,109770.71,22.82
3,resultado_neto,134330.50,141527.88,NaN,184909.99,189089.01,NaN


### Resultados de la regresión lineal

La regresión lineal se comparó con el baseline estacional utilizando las mismas cuatro observaciones de prueba correspondientes a 2025.

Los resultados muestran que el baseline estacional obtuvo menores errores para las cuatro variables financieras. En ingresos, el MAPE aumentó de 1,80 % a 7,03 %, mientras que en utilidad bruta pasó de 2,46 % a 9,32 %. Para el EBITDA recurrente, el MAPE aumentó de 17,86 % a 22,82 %.

En el resultado neto, el baseline también presentó menores valores de MAE y RMSE. Por tanto, la regresión lineal no demuestra una mejora predictiva frente a la referencia estacional en el período evaluado.

Este resultado sugiere que las características utilizadas en la regresión no aportaron suficiente información adicional para superar la referencia simple. Debido al tamaño reducido del conjunto de datos, esta conclusión es preliminar y se limita al período de prueba analizado.


### Implementación del modelo suavizamiento exponencial

In [11]:
%pip install statsmodels

   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   -- ------------------------------------- 0.8/11.3 MB 4.3 MB/s eta 0:00:03
   ---- ----------------------------------- 1.3/11.3 MB 3.1 MB/s eta 0:00:04
   ----- ---------------------------------- 1.6/11.3 MB 2.9 MB/s eta 0:00:04
   --------- ------------------------------ 2.6/11.3 MB 3.2 MB/s eta 0:00:03
   ------------ --------------------------- 3.4/11.3 MB 3.3 MB/s eta 0:00:03
   ------------ --------------------------- 3.7/11.3 MB 3.3 MB/s eta 0:00:03
   --------------- ------------------------ 4.5/11.3 MB 3.1 MB/s eta 0:00:03
   ---------------- ----------------------- 4.7/11.3 MB 2.8 MB/s eta 0:00:03
   ---------------- ----------------------- 4.7/11.3 MB 2.8 MB/s eta 0:00:03
   ----------------- ---------------------- 5.0/11.3 MB 2.5 MB/s eta 0:00:03
   ------------------ --------------------- 5.2/11.3 MB 2.4 MB/s eta 0:00:03
   ----------


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:

from statsmodels.tsa.holtwinters import SimpleExpSmoothing

resultados_suavizamiento = {}

for variable in variables_objetivo:

    # Serie de entrenamiento
    y_train = df_train[variable].astype(float)

    # Entrenar el modelo
    modelo = SimpleExpSmoothing(
        y_train,
        initialization_method="estimated"
    ).fit()

    # Pronosticar los cuatro trimestres de prueba
    y_pred = modelo.forecast(len(df_test))

    resultados_suavizamiento[variable] = {
        "modelo": modelo,
        "pronosticos": np.asarray(y_pred)
    }

    print(f"{variable}: modelo entrenado correctamente.")


ingresos: modelo entrenado correctamente.
utilidad_bruta: modelo entrenado correctamente.
ebitda_recurrente: modelo entrenado correctamente.
resultado_neto: modelo entrenado correctamente.


In [11]:
# Comparación de los valores reales y los estimados

comparacion_suavizamiento = df_test[["fecha"]].copy()

for variable in variables_objetivo:
    comparacion_suavizamiento[f"{variable}_real"] = (
        df_test[variable].to_numpy()
    )

    comparacion_suavizamiento[f"{variable}_suavizamiento"] = (
        resultados_suavizamiento[variable]["pronosticos"]
    )

display(comparacion_suavizamiento.round(2))


C:\Users\USUARIO\AppData\Local\Temp\ipykernel_19828\756245884.py:14: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(comparacion_suavizamiento.round(2))


,fecha,ingresos_real,ingresos_suavizamiento,utilidad_bruta_real,utilidad_bruta_suavizamiento,ebitda_recurrente_real,ebitda_recurrente_suavizamiento,resultado_neto_real,resultado_neto_suavizamiento
16,2025-03-31,5404642,5997933.72,1382773,1478560.52,371148,403799.83,93147,47147.25
17,2025-06-30,5208469,5997933.72,1335783,1478560.52,452242,403799.83,146865,47147.25
18,2025-09-30,5228905,5997933.72,1312056,1478560.52,448017,403799.83,142905,47147.25
19,2025-12-31,6166344,5997933.72,1612428,1478560.52,670615,403799.83,209191,47147.25


In [12]:
# Calculo de métricas del modelo de suavizamiento exponencial 

resultados_metricas_suavizamiento = []

for variable in variables_objetivo:
    y_real = df_test[variable].to_numpy()
    y_pred = resultados_suavizamiento[variable]["pronosticos"]

    mae = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))

    if variable != "resultado_neto":
        mape = np.mean(
            np.abs((y_real - y_pred) / y_real)
        ) * 100
    else:
        mape = np.nan

    resultados_metricas_suavizamiento.append({
        "variable": variable,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE_pct": mape
    })

metricas_suavizamiento = pd.DataFrame(
    resultados_metricas_suavizamiento
)

display(metricas_suavizamiento.round(2))


,variable,MAE,RMSE,MAPE_pct
0,ingresos,580048.86,631469.88,10.89
1,utilidad_bruta,134734.26,137117.58,9.65
2,ebitda_recurrente,98031.59,138345.84,17.29
3,resultado_neto,100879.75,108957.94,NaN


### Resultados del suavizamiento exponencial simple

El modelo de suavizamiento exponencial simple se evaluó sobre los cuatro trimestres de 2025.

Los resultados muestran que el modelo obtuvo un MAE de 100.879,75 para el resultado neto, inferior al registrado por el baseline estacional y la regresión lineal. Sin embargo, su desempeño fue menos favorable para los ingresos, la utilidad bruta y el EBITDA recurrente.

Estos resultados sugieren que el suavizamiento del nivel de la serie puede resultar útil para determinadas variables, pero no constituye una solución general para todas las métricas financieras analizadas.

La selección definitiva del modelo queda pendiente de una comparación bajo un esquema de evaluación temporal equivalente para todos los candidatos.

## 6. Evaluación temporal equivalente

Para comparar los modelos bajo las mismas condiciones, se realizará una evaluación temporal de un paso adelante sobre los cuatro trimestres de 2025.

En cada iteración, los modelos utilizarán únicamente la información disponible hasta el trimestre anterior al que se desea pronosticar. Una vez evaluado el trimestre, se incorporará su valor real para generar el pronóstico del siguiente período.

Se compararán el baseline estacional, la regresión lineal y el suavizamiento exponencial simple mediante MAE, RMSE y MAPE cuando esta última métrica resulte apropiada.

In [13]:
# Preparación de la evaluación


from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.holtwinters import SimpleExpSmoothing

# Copia ordenada de los datos con características
df_eval = df_model.sort_values("fecha").reset_index(drop=True).copy()

n_test = 4
inicio_test = len(df_eval) - n_test

print("Inicio de prueba:", df_eval.loc[inicio_test, "fecha"])
print("Fin de prueba:", df_eval.iloc[-1]["fecha"])
print("Número de trimestres de prueba:", n_test)


Inicio de prueba: 2025-03-31 00:00:00
Fin de prueba: 2025-12-31 00:00:00
Número de trimestres de prueba: 4


In [ ]:
# Generación de pronósticos comparables

columnas_estacionales = [
    col for col in ["T_2", "T_3", "T_4"]
    if col in df_eval.columns
]

pronosticos_temporales = []

for idx in range(inicio_test, len(df_eval)):
    historial = df_eval.iloc[:idx].copy()
    fila_objetivo = df_eval.iloc[idx]

    registro = {
        "fecha": fila_objetivo["fecha"]
    }

    for variable in variables_objetivo:
        # 1. Baseline estacional: mismo trimestre del año anterior
        registro[f"{variable}_real"] = fila_objetivo[variable]
        registro[f"{variable}_baseline"] = fila_objetivo[f"{variable}_lag4"]

        # 2. Regresión lineal, entrenada solo con el historial
        columnas_features = [
            "t",
            f"{variable}_lag1",
            f"{variable}_lag4"
        ] + columnas_estacionales

        X_train = historial[columnas_features]
        y_train = historial[variable]
        X_next = df_eval.loc[[idx], columnas_features]

        modelo_reg = LinearRegression()
        modelo_reg.fit(X_train, y_train)

        registro[f"{variable}_regresion"] = (
            modelo_reg.predict(X_next)[0]
        )

        # 3. Suavizamiento exponencial, reentrenado con el historial
        modelo_ses = SimpleExpSmoothing(
            historial[variable].astype(float),
            initialization_method="estimated"
        ).fit()

        registro[f"{variable}_suavizamiento"] = (
            modelo_ses.forecast(1).iloc[0]
        )

    pronosticos_temporales.append(registro)

df_pronosticos_temporales = pd.DataFrame(pronosticos_temporales)

display(df_pronosticos_temporales.round(2))


C:\Users\USUARIO\AppData\Local\Temp\ipykernel_19828\1303064465.py:56: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(df_pronosticos_temporales.round(2))


,fecha,ingresos_real,ingresos_baseline,ingresos_regresion,ingresos_suavizamiento,utilidad_bruta_real,utilidad_bruta_baseline,utilidad_bruta_regresion,utilidad_bruta_suavizamiento,ebitda_recurrente_real,ebitda_recurrente_baseline,ebitda_recurrente_regresion,ebitda_recurrente_suavizamiento,resultado_neto_real,resultado_neto_baseline,resultado_neto_regresion,resultado_neto_suavizamiento
0,2025-03-31,5404642,5275139.0,5939930.12,5997933.72,1382773,1321953.0,1588245.73,1478560.52,371148,302113.0,297271.38,403799.83,93147,-37863.0,-27243.32,47147.25
1,2025-06-30,5208469,5074917.0,5514621.61,5585497.50,1335783,1299704.0,1371288.48,1421691.03,452242,341931.0,358682.69,401870.71,146865,-18735.0,-10842.02,49862.37
2,2025-09-30,5228905,5242429.0,5452816.36,5363211.90,1312056,1286381.0,1397102.84,1381228.46,448017,342181.0,356909.26,404667.92,142905,-34733.0,80519.75,82199.00
3,2025-12-31,6166344,6288024.0,5918187.49,5285573.18,1612428,1624970.0,1618266.88,1348532.33,670615,638210.0,582552.71,406957.67,209191,146117.0,110102.95,59842.92


In [15]:
# Calculo de métricas

metricas_temporales = []

modelos_a_comparar = [
    "baseline",
    "regresion",
    "suavizamiento"
]

for variable in variables_objetivo:
    y_real = df_pronosticos_temporales[f"{variable}_real"].to_numpy()

    for nombre_modelo in modelos_a_comparar:
        y_pred = df_pronosticos_temporales[
            f"{variable}_{nombre_modelo}"
        ].to_numpy()

        mae = mean_absolute_error(y_real, y_pred)
        rmse = np.sqrt(mean_squared_error(y_real, y_pred))

        # MAPE solo para variables con valores reales positivos
        if variable != "resultado_neto" and np.all(y_real != 0):
            mape = np.mean(np.abs((y_real - y_pred) / y_real)) * 100
        else:
            mape = np.nan

        metricas_temporales.append({
            "variable": variable,
            "modelo": nombre_modelo,
            "MAE": mae,
            "RMSE": rmse,
            "MAPE_pct": mape
        })

metricas_temporales = pd.DataFrame(metricas_temporales)

display(
    metricas_temporales.sort_values(
        ["variable", "MAE"]
    ).round(2)
)


,variable,modelo,MAE,RMSE,MAPE_pct
6,ebitda_recurrente,baseline,79396.75,85419.10,17.86
7,ebitda_recurrente,regresion,86651.49,86986.62,18.52
8,ebitda_recurrente,suavizamiento,97507.38,136928.57,17.23
0,ingresos,baseline,99564.75,111350.89,1.80
1,ingresos,regresion,328377.15,350706.69,6.02
2,ingresos,suavizamiento,496349.49,567437.47,8.77
11,resultado_neto,suavizamiento,88264.12,96844.66,NaN
10,resultado_neto,regresion,109892.66,115190.81,NaN
9,resultado_neto,baseline,134330.50,141527.88,NaN
3,utilidad_bruta,baseline,33779.00,38135.53,2.46


### Resultados de la evaluación temporal equivalente

Se compararon el baseline estacional, la regresión lineal y el suavizamiento exponencial simple mediante una evaluación de un paso adelante sobre los cuatro trimestres de 2025.

El baseline estacional obtuvo los menores valores de MAE y RMSE para los ingresos, la utilidad bruta y el EBITDA recurrente. Para el resultado neto, el suavizamiento exponencial simple presentó el menor error absoluto y cuadrático.

Estos resultados sugieren que una referencia estacional sencilla constituye una alternativa competitiva para tres de las cuatro variables analizadas. No obstante, la selección es preliminar, ya que se basa en solo cuatro observaciones de prueba.

Como siguiente paso, se evaluará un modelo Holt-Winters con tendencia y estacionalidad aditivas para determinar si la incorporación explícita de estos componentes mejora los pronósticos.

### Modelo Holt-Winters

In [16]:

from statsmodels.tsa.holtwinters import ExponentialSmoothing

resultados_holt_winters = []

for idx in range(inicio_test, len(df_eval)):
    historial = df_eval.iloc[:idx].copy()
    fila_objetivo = df_eval.iloc[idx]

    registro = {
        "fecha": fila_objetivo["fecha"]
    }

    for variable in variables_objetivo:
        y_train = historial[variable].astype(float)

        try:
            modelo_hw = ExponentialSmoothing(
                y_train,
                trend="add",
                seasonal="add",
                seasonal_periods=4,
                initialization_method="estimated"
            ).fit(optimized=True)

            pronostico = float(modelo_hw.forecast(1).iloc[0])
            estado = "correcto"

        except Exception as error:
            pronostico = np.nan
            estado = str(error)

        registro[f"{variable}_real"] = fila_objetivo[variable]
        registro[f"{variable}_holt_winters"] = pronostico
        registro[f"{variable}_estado"] = estado

    resultados_holt_winters.append(registro)

df_holt_winters = pd.DataFrame(resultados_holt_winters)

display(df_holt_winters.round(2))


C:\Users\USUARIO\AppData\Local\Temp\ipykernel_19828\762759579.py:40: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(df_holt_winters.round(2))


,fecha,ingresos_real,ingresos_holt_winters,ingresos_estado,utilidad_bruta_real,utilidad_bruta_holt_winters,utilidad_bruta_estado,ebitda_recurrente_real,ebitda_recurrente_holt_winters,ebitda_recurrente_estado,resultado_neto_real,resultado_neto_holt_winters,resultado_neto_estado
0,2025-03-31,5404642,5600580.58,correcto,1382773,1447565.30,correcto,371148,349179.80,correcto,93147,-37221.45,correcto
1,2025-06-30,5208469,5282063.14,correcto,1335783,1345786.20,correcto,452242,371159.77,correcto,146865,-26597.88,correcto
2,2025-09-30,5228905,5526714.64,correcto,1312056,1351670.84,correcto,448017,431929.58,correcto,142905,82299.54,correcto
3,2025-12-31,6166344,6044237.84,correcto,1612428,1549987.73,correcto,670615,635809.95,correcto,209191,245696.24,correcto


In [17]:
# Calculo de métricas

metricas_holt_winters = []

for variable in variables_objetivo:
    y_real = df_holt_winters[f"{variable}_real"].to_numpy()
    y_pred = df_holt_winters[f"{variable}_holt_winters"].to_numpy()

    validos = np.isfinite(y_real) & np.isfinite(y_pred)
    y_real = y_real[validos]
    y_pred = y_pred[validos]

    if len(y_real) == 0:
        continue

    mae = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))

    if variable != "resultado_neto" and np.all(y_real != 0):
        mape = np.mean(np.abs((y_real - y_pred) / y_real)) * 100
    else:
        mape = np.nan

    metricas_holt_winters.append({
        "variable": variable,
        "modelo": "holt_winters",
        "MAE": mae,
        "RMSE": rmse,
        "MAPE_pct": mape,
        "observaciones_evaluadas": len(y_real)
    })

metricas_holt_winters = pd.DataFrame(metricas_holt_winters)

display(metricas_holt_winters.round(2))


,variable,modelo,MAE,RMSE,MAPE_pct,observaciones_evaluadas
0,ingresos,holt_winters,172362.13,191969.09,3.18,4
1,utilidad_bruta,holt_winters,44212.65,49412.12,3.08,4
2,ebitda_recurrente,holt_winters,38485.73,46171.22,8.16,4
3,resultado_neto,holt_winters,100235.51,114117.22,NaN,4


In [18]:

metricas_completas = pd.concat(
    [
        metricas_temporales,
        metricas_holt_winters[
            ["variable", "modelo", "MAE", "RMSE", "MAPE_pct"]
        ]
    ],
    ignore_index=True
)

display(
    metricas_completas
    .sort_values(["variable", "MAE"])
    .round(2)
)


,variable,modelo,MAE,RMSE,MAPE_pct
14,ebitda_recurrente,holt_winters,38485.73,46171.22,8.16
6,ebitda_recurrente,baseline,79396.75,85419.10,17.86
7,ebitda_recurrente,regresion,86651.49,86986.62,18.52
8,ebitda_recurrente,suavizamiento,97507.38,136928.57,17.23
0,ingresos,baseline,99564.75,111350.89,1.80
12,ingresos,holt_winters,172362.13,191969.09,3.18
1,ingresos,regresion,328377.15,350706.69,6.02
2,ingresos,suavizamiento,496349.49,567437.47,8.77
11,resultado_neto,suavizamiento,88264.12,96844.66,NaN
15,resultado_neto,holt_winters,100235.51,114117.22,NaN


In [19]:
#Mejor modelo por variable

mejores_modelos = (
    metricas_completas
    .sort_values("MAE")
    .groupby("variable", as_index=False)
    .first()
)

display(mejores_modelos.round(2))


,variable,modelo,MAE,RMSE,MAPE_pct
0,ebitda_recurrente,holt_winters,38485.73,46171.22,8.16
1,ingresos,baseline,99564.75,111350.89,1.80
2,resultado_neto,suavizamiento,88264.12,96844.66,NaN
3,utilidad_bruta,baseline,33779.00,38135.53,2.46


### Resultados de Holt-Winters

El modelo Holt-Winters, con tendencia y estacionalidad aditivas, se evaluó mediante pronósticos de un paso adelante para los cuatro trimestres de 2025.

El modelo presentó un mejor desempeño que el baseline estacional para el EBITDA recurrente y el resultado neto, según las métricas MAE y RMSE. En cambio, el baseline mantuvo menores errores para los ingresos y la utilidad bruta.

Los resultados sugieren que las variables financieras no necesariamente requieren el mismo modelo de pronóstico. La incorporación explícita de tendencia y estacionalidad parece aportar valor para algunas series, pero no para todas.

La selección de modelos debe considerarse preliminar, dado que la evaluación comprende únicamente cuatro observaciones. Será necesario verificar su estabilidad con más períodos de evaluación antes de recomendar su uso para pronósticos futuros.

### Selección de modelos para el pronóstico financiero

Se evaluaron cuatro enfoques de pronóstico para las variables financieras de Grupo Éxito: baseline estacional, regresión lineal, suavizamiento exponencial simple y Holt-Winters.

Los resultados permitieron seleccionar el modelo con menor error para cada variable. El baseline estacional obtuvo el mejor desempeño para los ingresos y la utilidad bruta, con MAPE de 1,80 % y 2,46 %, respectivamente. Para el EBITDA recurrente, Holt-Winters presentó un MAE de 38.485,73 y un MAPE de 8,16 %. Por su parte, el suavizamiento exponencial simple obtuvo el menor MAE para el resultado neto, con un valor de 88.264,12.

La selección de modelos específicos por variable responde a las diferencias en sus patrones temporales y niveles de variabilidad. Los resultados sugieren que los modelos sencillos pueden ser competitivos y que la incorporación de tendencia y estacionalidad explícitas puede beneficiar determinadas series.

Sin embargo, la evaluación se realizó sobre cuatro trimestres de prueba, correspondientes a 2025. Por tanto, los resultados deben interpretarse como evidencia preliminar de desempeño y no como una garantía de precisión futura. Se recomienda complementar el análisis con validación temporal en varios períodos y documentar las limitaciones derivadas del tamaño de la muestra.